# 02 — Baseline assignment (run + report)

Run a static user-equilibrium assignment on Chicago Sketch, then read the results: convergence, system MOE, and the fused HTML report. Every run passes the intake gate and writes a reproducibility **manifest**.

In [1]:
import os
from dtalite_qa.api import Network, Demand, Scenario, AssignmentEngine
EXE = 'cmake_build_rel/DTALite_exe.exe'
net = Network.read_gmns('kernel/data_sets/03_chicago_sketch')
scen = Scenario(net, Demand.from_network(net),
                settings={'iterations': 40, 'gap_tolerance': 0})
result = AssignmentEngine().run(scen, exe=EXE)
result.ok

True

## Convergence
Frank–Wolfe drives the **relative gap** toward zero. Each iteration's gap:

In [2]:
traj = result.convergence()
for g in traj[::5] + traj[-1:]:
    print(f"iter {g['iter']:3d}   gap {g['gap_pct']:8.4f}%   system VMT {g['system_vmt']:,.0f}")
print('final gap %:', result.final_gap_pct())

iter   1   gap 510.7808%   system VMT 0
iter   6   gap   1.5083%   system VMT 12,218,884
iter  11   gap   0.6295%   system VMT 12,196,201
iter  16   gap   0.1953%   system VMT 12,173,191
iter  21   gap   0.1755%   system VMT 12,166,635
iter  26   gap   0.0879%   system VMT 12,160,906
iter  31   gap   0.0523%   system VMT 12,160,146
iter  36   gap   0.0530%   system VMT 12,158,723
iter  39   gap   0.0386%   system VMT 12,158,476
final gap %: 0.038647


## System MOE
Vehicle-miles / hours travelled, mean network speed, and how many links carry load.

In [3]:
result.moe()

{'links': 2950,
 'loaded_links': 2928,
 'vmt': 36475322.9,
 'vht': 918963.5,
 'mean_speed_mph': 39.69}

## Busiest links
Sort loaded links by volume; `doc` is the volume/capacity ratio (V/C).

In [4]:
links = result.link_volumes()
from dtalite_qa import csvio
top = sorted(links, key=lambda r: -csvio.fnum(r.get('volume')))[:8]
for r in top:
    print(f"link {r.get('link_id'):>6}  vol {csvio.fnum(r.get('volume')):9,.0f}  "
          f"V/C {csvio.fnum(r.get('doc')):.2f}  {csvio.fnum(r.get('speed_mph') or r.get('speed')):5.1f} mph")

link   1071  vol    22,381  V/C 0.45    0.0 mph
link   1077  vol    22,237  V/C 0.45    0.0 mph
link   1084  vol    20,209  V/C 0.84   39.3 mph
link   1009  vol    19,525  V/C 0.98   38.1 mph
link   1087  vol    19,092  V/C 1.36   27.1 mph
link   1081  vol    18,425  V/C 0.77   40.2 mph
link   1082  vol    17,801  V/C 0.36    0.0 mph
link      5  vol    17,224  V/C 0.35    0.0 mph


## The fused HTML report
One self-contained page: network/OD summary, convergence, V/C histogram, the assignment map, top links, assigned-vs-reference scatter, and the reproducibility manifest. No external assets.

In [5]:
from dtalite_qa import report_html
html = report_html.build_report(result.run_dir, project_name='Chicago Sketch baseline')
print('open this in a browser:', html)
print('size:', os.path.getsize(html), 'bytes  | self-contained (no CDN)')

open this in a browser: C:\Users\xzhou\AppData\Local\Temp\dtalite_run_opmkju7y\report.html
size: 430147 bytes  | self-contained (no CDN)


## Reproducibility manifest
Every run records the exact inputs (sha256), effective settings, kernel identity, the intake-gate status, and the MOE — so two runs are comparable and a result is auditable.

In [6]:
m = result.manifest
print('gate:', m.get('intake_gate'))
print('kernel sha256:', (m.get('exe') or {}).get('sha256','')[:16], '...')
print('iterations:', m['convergence']['iterations'], '| final gap %:', m['convergence']['final_gap_pct'])
print('input files hashed:', list(m['files'])[:6], '...')

gate: READY
kernel sha256: 3c2c9143859f2a9e ...
iterations: 39 | final gap %: 0.038647
input files hashed: ['TAP_log.csv', 'demand.csv', 'destination_accessibility.csv', 'google_maps_od_distance.csv', 'inaccessible_od.csv', 'link.csv'] ...


## The same thing from the command line
Analysts who don't write Python get the identical pipeline from one config file:
```bash
taplite run configs/chicago_sketch_baseline.yml   # gate → kernel → manifest → report.html
taplite report runs/chicago_baseline              # (re)generate the HTML
taplite compare runs/chicago_baseline runs/other  # side-by-side MOE diff
```